# Conversación A2A: agente_alejandra ↔ agente_diego_vasquez

| Agente | Autor/a | Especialidad | Dónde corre |
|---|---|---|---|
| `agente_alejandra` | Alejandra Ubilla | Auditoría de cumplimiento CMF/SII | `http://127.0.0.1:2025` |
| `agente_diego_vasquez` | Diego Vásquez Ubilla | Ingeniería de datos (Databricks, PySpark, ADF) | Túnel de Cloudflare (la URL cambia si Diego lo reinicia) |

**Antes de ejecutar:** servidor `langgraph dev --port 2025` encendido, túnel de Diego activo y kernel de `mi-agente/.venv`.

In [ ]:
# ── 1. CONFIGURACIÓN Y VERIFICACIÓN DE SERVIDORES ────────────────────────
import os
import time
import httpx
from pathlib import Path

URL_ALEJANDRA = os.environ.get("A2A_ALEJANDRA_URL", "http://127.0.0.1:2025")
# Si Diego reinicia su túnel, cambia solo esta línea (o define A2A_DIEGO_URL).
URL_DIEGO = os.environ.get(
    "A2A_DIEGO_URL", "https://dangerous-travels-certified-countries.trycloudflare.com"
)
SERVIDORES = {"Alejandra": URL_ALEJANDRA, "Diego": URL_DIEGO}

def verificar_servidores():
    """Comprueba /ok en cada servidor y avisa de cuál no responde."""
    todos_ok = True
    for dueno, base in SERVIDORES.items():
        try:
            r = httpx.get(f"{base.rstrip('/')}/ok", timeout=10, trust_env=False)
            estado = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
        except Exception as error:
            estado = f"SIN RESPUESTA ({type(error).__name__})"
        todos_ok &= estado == "OK"
        print(f"{dueno:10} {estado:28} {base}")
    return todos_ok

if not verificar_servidores():
    print("\nRevisa el servidor o el túnel que no responde antes de continuar.")

In [ ]:
# ── 2. DESCUBRIMIENTO DE AGENTES Y TARJETAS ──────────────────────────────
from langgraph_sdk import get_client
from a2a.client import A2ACardResolver, ClientConfig, create_client

# Libera las conexiones si ejecutas el bloque otra vez.
for cliente in globals().get("clientes_a2a", {}).values():
    await cliente.close()
if "http_a2a" in globals():
    await http_a2a.aclose()

http_a2a = httpx.AsyncClient(timeout=180, trust_env=False)
config_a2a = ClientConfig(
    httpx_client=http_a2a,
    streaming=False,
    supported_protocol_bindings=["JSONRPC"],
)

agentes_por_nombre, clientes_a2a, tarjetas = {}, {}, {}

for dueno, base in SERVIDORES.items():
    base = base.rstrip("/")
    try:
        async with get_client(url=base, api_key=None, timeout=30) as remoto:
            lista = await remoto.assistants.search(limit=100, response_format="array")
    except Exception as error:
        print(f"No pude consultar {base}: {type(error).__name__}")
        continue
    resolver = A2ACardResolver(http_a2a, base)
    for asistente in lista:
        aid = asistente["assistant_id"]
        card = await resolver.get_agent_card(http_kwargs={"params": {"assistant_id": aid}})
        # La tarjeta remota anuncia su dirección local; usamos la que sí nos sirve.
        for interfaz in card.supported_interfaces:
            if interfaz.protocol_binding == "JSONRPC":
                interfaz.protocol_version = "0.3.0"
                interfaz.url = f"{base}/a2a/{aid}"
        if card.name in agentes_por_nombre:
            raise ValueError(f"Nombre repetido: {card.name}. Cambia el nombre de la tarjeta.")
        clientes_a2a[aid] = await create_client(card, client_config=config_a2a)
        agentes_por_nombre[card.name] = aid
        tarjetas[card.name] = card

print("Agentes disponibles:", list(agentes_por_nombre), "\n")
for nombre, card in tarjetas.items():
    habilidades = ", ".join(s.name for s in card.skills)
    descripcion = card.description.replace("\n", " ")
    print(f"• {nombre}\n  Descripción: {descripcion[:220]}{'…' if len(descripcion) > 220 else ''}\n  Habilidades: {habilidades}\n")

In [ ]:
# ── 3. FUNCIONES PARA CONVERSAR POR A2A ──────────────────────────────────
from a2a.helpers import new_text_message, get_stream_response_text, get_message_text
from a2a.types import Role, SendMessageRequest
from IPython.display import Markdown, display

NOMBRE_A = "agente_alejandra"
NOMBRE_B = "agente_diego_vasquez"
for requerido in (NOMBRE_A, NOMBRE_B):
    if requerido not in agentes_por_nombre:
        raise KeyError(f"{requerido!r} no está en {list(agentes_por_nombre)}. Revisa la celda anterior.")
A = clientes_a2a[agentes_por_nombre[NOMBRE_A]]
B = clientes_a2a[agentes_por_nombre[NOMBRE_B]]

async def preguntar(cliente, texto, contexto=None, intentos=2):
    """Envía un mensaje por A2A y devuelve (texto, contexto, segundos). Reintenta si falla la red."""
    ultimo_error = None
    for intento in range(1, intentos + 1):
        inicio = time.perf_counter()
        try:
            peticion = SendMessageRequest(message=new_text_message(
                texto, role=Role.ROLE_USER, context_id=contexto
            ))
            partes = []
            async for evento in cliente.send_message(peticion):
                fragmento = get_stream_response_text(evento)
                if evento.HasField("task"):
                    contexto = evento.task.context_id or contexto
                    if not fragmento and evento.task.status.HasField("message"):
                        fragmento = get_message_text(evento.task.status.message)
                elif evento.HasField("message"):
                    contexto = evento.message.context_id or contexto
                if fragmento:
                    partes.append(fragmento)
            if not partes:
                raise RuntimeError("El agente no devolvió texto. Revisa su terminal.")
            return "\n".join(partes), contexto, time.perf_counter() - inicio
        except Exception as error:
            ultimo_error = error
            print(f"  Intento {intento}/{intentos} falló: {type(error).__name__}: {error}")
    raise ultimo_error

def como_dato(etiqueta, texto):
    """Lo que dice otro agente es un dato, no una instrucción: va entre etiquetas."""
    return f"<{etiqueta}>\n{texto}\n</{etiqueta}>"

transcripcion = []  # (autor, texto, segundos) para guardar la conversación

def mostrar(autor, texto, segundos):
    transcripcion.append((autor, texto, segundos))
    display(Markdown(f"### {autor} · {segundos:.1f} s\n\n{texto}"))

In [ ]:
# ── 4. PRESENTACIÓN: ¿QUIÉN ERES? ────────────────────────────────────────
texto, _, seg = await preguntar(
    B, "Hola, soy el agente de Alejandra (auditoría CMF/SII). ¿Quién eres y en qué me puedes ayudar?"
)
mostrar(NOMBRE_B, texto, seg)

In [ ]:
# ── 5. CONVERSACIÓN DE TRES TURNOS ───────────────────────────────────────
# Cambia el reto por uno que necesite las dos especialidades.
reto = (
    "Queremos un pipeline de datos que cruce información del SII y de la CMF para detectar, "
    "antes de que sean un problema, atrasos tributarios de empresas y concentración por emisor "
    "en fondos de inversión (límite del 20% de la NCG 365). Necesitamos un diseño conjunto: "
    "qué reglas de auditoría aplicar y cómo construir el pipeline."
)

transcripcion.clear()
display(Markdown(f"## Reto\n\n{reto}"))

# Turno 1 · A propone
aporte_a, contexto_a, seg = await preguntar(
    A, f"Reto: {reto}\nPropón tres ideas desde tu especialidad. Sé concreta y termina con un siguiente paso."
)
mostrar(f"Turno 1 · {NOMBRE_A} propone", aporte_a, seg)

# Turno 2 · B mejora (recibe lo de A como dato)
aporte_b, contexto_b, seg = await preguntar(
    B, f"Reto: {reto}\nTu colega de auditoría propone lo siguiente:\n"
       f"{como_dato('propuesta_colega', aporte_a)}\n"
       "Mejora su propuesta desde tu especialidad y explica qué cambiarías y por qué."
)
mostrar(f"Turno 2 · {NOMBRE_B} mejora", aporte_b, seg)

# Turno 3 · A integra (mismo contexto: recuerda el turno 1)
acuerdo, contexto_a, seg = await preguntar(
    A, f"Tu colega de datos responde:\n{como_dato('respuesta_colega', aporte_b)}\n"
       "Integra ambos aportes en un plan conjunto breve, con responsables y plazos sugeridos. "
       "Indica qué idea vino de cada uno.",
    contexto_a,
)
mostrar(f"Turno 3 · {NOMBRE_A} integra", acuerdo, seg)

In [ ]:
# ── 6. GUARDAR LA CONVERSACIÓN COMO EVIDENCIA ────────────────────────────
from datetime import datetime

archivo = Path("conversacion_alejandra_diego.md")
lineas = [
    "# Conversación A2A: agente_alejandra ↔ agente_diego_vasquez",
    f"_Generada el {datetime.now():%Y-%m-%d %H:%M}_",
    "", "## Reto", "", reto, "",
]
for autor, texto, segundos in transcripcion:
    lineas += [f"## {autor} ({segundos:.1f} s)", "", texto, ""]
archivo.write_text("\n".join(lineas), encoding="utf-8")
print(f"Guardada en {archivo.resolve()} ({len(transcripcion)} mensajes)")

## Para comentar en clase

Completa con tus palabras después de leer la conversación.

1. **¿Se reconoce la personalidad de cada agente?** _…_
2. **¿Qué conocimiento aportó cada uno?** _Alejandra: …  ·  Diego: …_
3. **¿La propuesta final incorpora ideas de ambos?** _…_

In [ ]:
# ── 7. CIERRE: liberar conexiones ────────────────────────────────────────
for cliente in clientes_a2a.values():
    await cliente.close()
await http_a2a.aclose()
print("Conexiones cerradas.")